## GABM Ablation Study for Evacuation Modelling

### Objective
This notebook tests whether performance gains come from:
1. evacuation mechanics,
2. behavioral modelling in general,
3. or specifically full GABM integration.

We use the same NTU project data and run paired multi-seed experiments for fair comparison.



**Environment Setup**

This section imports dependencies, finds the repo root robustly (so folder moves do not break paths), and loads simulation classes.



In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score


# Find repo root by searching current folder and parent folders.
def find_repo_path(repo_name='FYP-Generative-Agent-Evacuation-Simulation'):
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        candidate = p / repo_name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find '{repo_name}' from: {start}")


repo_root = find_repo_path()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pysocialforce as psf
from pysocialforce.sceneforevacuation import PedStateEvacuation, EnvStateEvacuation

print('Using repo root:', repo_root)
print('pysocialforce loaded from:', psf.__file__)



**Experimental Design**

### Variants
1. **SFM-Core**: base social-force simulator without project evacuation state class.
2. **Evac-Mechanics Baseline**: project baseline (`PedStateEvacuation`) without behavioral grid.
3. **Surrogate-Behavior**: behavior-driven simulation using a supervised surrogate labeler trained on the same behavior grid.
4. **Full-GABM**: behavior-driven simulation using the original behavior grid labels.

### Fairness controls
- Same seed is used across all variants in one batch.
- Same room geometry, agent count, and runtime budget.
- Same initial positions and goals per seed.



In [ ]:
# Core experiment settings
N_AGENTS = 50
TAU = 0.5
MAX_STEPS = 3000
SEEDS = list(range(100, 105))  # 5 seeds by default for faster turnaround

ROOM_SIZE = 20.0
DOOR_WIDTH = 1.2
DOOR_X = ROOM_SIZE / 2
DOOR_Y = 0.0

CFG = repo_root / 'experiments' / 'example.toml'
BEHAVIOR_JSON = repo_root / 'behavior grid analysis' / 'behavior impatience analysis' / 'behavior_grid.json'

print('Seeds:', SEEDS)
print('Config file:', CFG)
print('Behavior grid:', BEHAVIOR_JSON)



**Shared State Generation**

Each seed generates one shared crowd scenario so model comparisons are paired and fair.



In [ ]:
def create_room(size=20.0, door_width=1.2):
    # Create square room with a single door opening on the right wall.
    h = door_width / 2
    return [
        [-size/2, size/2, -size/2, -size/2],
        [-size/2, size/2,  size/2,  size/2],
        [-size/2, -size/2, -size/2, size/2],
        [ size/2,  size/2, -size/2, -h],
        [ size/2,  size/2,  h, size/2],
    ]


def generate_positions(rng, n_agents, room_size=20.0, margin=2.0):
    # Random starting positions with wall margin.
    return rng.uniform(-room_size/2 + margin, room_size/2 - margin, size=(n_agents, 2))


def generate_characteristics(rng, n_agents):
    # Generate demographic + OCEAN trait levels to match behavior grid schema.
    ages = ['Young Adult', 'Adult', 'Middle Age']
    genders = ['male', 'female']
    levels = ['Low', 'High']
    out = []
    for _ in range(n_agents):
        out.append([rng.choice(ages), rng.choice(genders)] + [rng.choice(levels) for _ in range(5)])
    return out


def build_initial_states(seed, n_agents=50, tau=0.5, room_size=20.0):
    # Build paired baseline and behavior-capable initial states from one seed.
    rng = np.random.default_rng(seed)

    door_center = np.array([room_size / 2, 0.0])
    final_goal = door_center + np.array([3.0, 0.0])

    pos0 = generate_positions(rng, n_agents, room_size)
    vel0 = np.zeros((n_agents, 2), dtype=float)

    for i in range(n_agents):
        d = final_goal - pos0[i]
        vel0[i] = (d / np.linalg.norm(d)) * 0.5

    goals = np.tile(final_goal, (n_agents, 1))

    # SFM/baseline style state [px,py,vx,vy,gx,gy]
    base_state = np.zeros((n_agents, 6), dtype=float)
    base_state[:, :2] = pos0
    base_state[:, 2:4] = vel0
    base_state[:, 4:6] = goals

    # Behavior-capable state [px,py,vx,vy,gx,gy,tau,age,gender,O,C,E,A,N]
    behv_state = np.empty((n_agents, 14), dtype=object)
    behv_state[:, :2] = pos0
    behv_state[:, 2:4] = vel0
    behv_state[:, 4:6] = goals
    behv_state[:, 6] = tau

    chars = generate_characteristics(rng, n_agents)
    for i, c in enumerate(chars):
        behv_state[i, 7:] = c

    return base_state, behv_state, final_goal



**Evaluation Metrics**

We evaluate:
- `total_time_s`: total evacuation duration
- `t25_s`: time to 25% evacuated (early-phase speed)
- `congestion_index`: mean active agents near door zone

Lower values are generally better for all three metrics.



In [ ]:
def time_to_fraction(escaped_counts, times, n_agents, frac=0.25):
    target = int(np.ceil(n_agents * frac))
    idx = np.where(np.array(escaped_counts) >= target)[0]
    return float(times[idx[0]]) if len(idx) else np.nan


def summarize_metrics(times, escaped_counts, near_exit_counts, n_agents):
    return {
        'total_time_s': float(times[-1]),
        't25_s': time_to_fraction(escaped_counts, times, n_agents, frac=0.25),
        'congestion_index': float(np.mean(near_exit_counts)) if near_exit_counts else np.nan,
        'escaped_final': int(escaped_counts[-1]),
    }



**Surrogate Behavior Model (Non-LLM Labeler)**

To test whether LLM-generated labeling is uniquely necessary, we train a surrogate classifier on the same behavior grid and replace labels with surrogate predictions.

Important interpretation note:
- This does **not** replace true external validation.
- It only tests whether a simpler learned mapper can approximate behavior labels already present in the NTU grid.



In [ ]:
def build_surrogate_behavior_grid(behavior_json_path):
    # Train a surrogate classifier and return predicted-label behavior grid.
    data = json.loads(Path(behavior_json_path).read_text(encoding='utf-8'))
    df = pd.DataFrame(data)

    # Expand nested dict columns
    for trait in ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']:
        df[trait] = df['traits'].apply(lambda x: x[trait])

    df['fullness'] = df['condition'].apply(lambda x: x['fullness'])
    df['distance_to_exit'] = df['condition'].apply(lambda x: x['distance_to_exit'])

    # Encode categorical predictors
    features = ['age_category', 'gender', 'Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism', 'fullness', 'distance_to_exit']
    X = pd.DataFrame(index=df.index)

    for col in features:
        le = LabelEncoder()
        X[col] = le.fit_transform(df[col])

    y = (df['state_of_impatience'] == 'Yes').astype(int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'Surrogate holdout accuracy: {acc:.3f}')

    # Predict for all rows and rebuild behavior grid with predicted labels
    y_all = model.predict(X)
    predicted_grid = []

    for i, row in df.iterrows():
        entry = {
            'age_category': row['age_category'],
            'gender': row['gender'],
            'traits': row['traits'],
            'condition': row['condition'],
            'state_of_impatience': 'Yes' if y_all[i] == 1 else 'No',
        }
        predicted_grid.append(entry)

    return predicted_grid, acc


surrogate_grid, surrogate_acc = build_surrogate_behavior_grid(BEHAVIOR_JSON)
print('Surrogate grid size:', len(surrogate_grid))



**Simulation Runners for Each Variant**

This section defines one shared simulation loop and four variant wrappers.

Why one loop?
- Ensures identical metric logic across models.
- Reduces implementation bias in comparisons.



In [ ]:
def run_loop(sim, n_agents, final_goal_vec, has_native_escaped=True, tau=0.5, max_steps=3000, door_x=10.0, door_y=0.0):
    # Run one simulation and return standard metrics.
    # has_native_escaped=True for project evacuation classes exposing sim.peds.escaped.
    escaped_counts = [0]
    times = [0.0]
    near_exit_counts = []

    # For models without native escaped tracking, maintain our own mask.
    escaped_mask = np.zeros(n_agents, dtype=bool)

    for step in range(1, max_steps + 1):
        sim.step(1)
        pos = sim.peds.pos()

        if has_native_escaped:
            escaped_mask = sim.peds.escaped.copy()
        else:
            # Geometric escape rule for SFM-core comparison.
            past_door = pos[:, 0] > (door_x + 0.2)
            near_door_lane = np.abs(pos[:, 1] - door_y) <= 1.0
            near_goal = np.linalg.norm(pos - final_goal_vec.reshape(1, 2), axis=1) < 1.0
            escaped_mask = escaped_mask | (past_door & near_door_lane) | near_goal

        escaped = int(np.sum(escaped_mask))
        escaped_counts.append(escaped)
        times.append(step * tau)

        active = ~escaped_mask
        near_exit = (np.abs(pos[:, 0] - door_x) <= 1.0) & (np.abs(pos[:, 1] - door_y) <= 1.5) & active
        near_exit_counts.append(int(np.sum(near_exit)))

        if escaped >= n_agents:
            break

    return summarize_metrics(times, escaped_counts, near_exit_counts, n_agents)


def run_variant_sfm_core(base_state, obstacles, final_goal_vec):
    # Variant 1: SFM core with default simulator.
    sim = psf.Simulator(
        base_state,
        groups=None,
        obstacles=obstacles,
        config_file=CFG,
    )
    return run_loop(sim, base_state.shape[0], final_goal_vec, has_native_escaped=False, tau=TAU, max_steps=MAX_STEPS, door_x=DOOR_X, door_y=DOOR_Y)


def run_variant_evac_mechanics(base_state, obstacles):
    # Variant 2: Project evacuation mechanics baseline (no behavior grid).
    sim = psf.Simulator(
        base_state,
        groups=None,
        obstacles=obstacles,
        config_file=CFG,
        ped_state_class=PedStateEvacuation,
        env_state_class=EnvStateEvacuation,
    )
    return run_loop(sim, base_state.shape[0], base_state[0, 4:6], has_native_escaped=True, tau=TAU, max_steps=MAX_STEPS, door_x=DOOR_X, door_y=DOOR_Y)


def run_variant_behavior(behv_state, obstacles, mode='full'):
    # Variant 3/4: behavior-enabled simulator.
    # mode='full' uses original behavior grid; mode='surrogate' swaps in surrogate labels.
    sim = psf.SimulatorWithBehavior(
        behv_state,
        groups=None,
        obstacles=obstacles,
        config_file=CFG,
        behavior_file=BEHAVIOR_JSON,
    )

    if mode == 'surrogate':
        sim.behavior_grid = surrogate_grid
        sim.peds.behavior_grid = surrogate_grid

    return run_loop(sim, behv_state.shape[0], behv_state[0, 4:6].astype(float), has_native_escaped=True, tau=TAU, max_steps=MAX_STEPS, door_x=DOOR_X, door_y=DOOR_Y)



**Multi-Seed Ablation Execution**

Each seed generates one shared initial state, then all four variants are executed.
This gives paired per-seed comparisons.



In [ ]:
rows = []
obstacles = create_room(ROOM_SIZE, DOOR_WIDTH)

for seed in SEEDS:
    base_state, behv_state, final_goal = build_initial_states(seed, n_agents=N_AGENTS, tau=TAU, room_size=ROOM_SIZE)

    # Use .copy() so each variant starts from identical state and no cross-run mutation leaks.
    r_core = run_variant_sfm_core(base_state.copy(), obstacles, final_goal)
    r_evac = run_variant_evac_mechanics(base_state.copy(), obstacles)
    r_surr = run_variant_behavior(behv_state.copy(), obstacles, mode='surrogate')
    r_gabm = run_variant_behavior(behv_state.copy(), obstacles, mode='full')

    row = {
        'seed': seed,

        'core_total_time_s': r_core['total_time_s'],
        'core_t25_s': r_core['t25_s'],
        'core_congestion': r_core['congestion_index'],

        'evac_total_time_s': r_evac['total_time_s'],
        'evac_t25_s': r_evac['t25_s'],
        'evac_congestion': r_evac['congestion_index'],

        'surr_total_time_s': r_surr['total_time_s'],
        'surr_t25_s': r_surr['t25_s'],
        'surr_congestion': r_surr['congestion_index'],

        'gabm_total_time_s': r_gabm['total_time_s'],
        'gabm_t25_s': r_gabm['t25_s'],
        'gabm_congestion': r_gabm['congestion_index'],
    }

    # Example deltas versus project baseline (evac mechanics).
    row['delta_t25_gabm_vs_evac_pct'] = ((row['evac_t25_s'] - row['gabm_t25_s']) / row['gabm_t25_s'] * 100.0) if row['gabm_t25_s'] > 0 else np.nan
    row['delta_cong_gabm_vs_evac_pct'] = ((row['gabm_congestion'] - row['evac_congestion']) / row['evac_congestion'] * 100.0) if row['evac_congestion'] > 0 else np.nan

    rows.append(row)

results_df = pd.DataFrame(rows)
print('Per-seed results:')
display(results_df)



**Summary Tables and Visuals**

This section provides aggregate statistics and boxplots for quick comparison.



In [ ]:
summary_cols = [
    'core_t25_s', 'evac_t25_s', 'surr_t25_s', 'gabm_t25_s',
    'core_congestion', 'evac_congestion', 'surr_congestion', 'gabm_congestion',
    'core_total_time_s', 'evac_total_time_s', 'surr_total_time_s', 'gabm_total_time_s',
]

summary = results_df[summary_cols].describe().T[['mean', 'std', 'min', 'max']]
print('Summary statistics:')
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([
    results_df['core_t25_s'].dropna(),
    results_df['evac_t25_s'].dropna(),
    results_df['surr_t25_s'].dropna(),
    results_df['gabm_t25_s'].dropna(),
], labels=['SFM-Core', 'Evac-Baseline', 'Surrogate', 'Full-GABM'])
axes[0].set_title('T25 (lower is better)')
axes[0].set_ylabel('Seconds')
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].boxplot([
    results_df['core_congestion'].dropna(),
    results_df['evac_congestion'].dropna(),
    results_df['surr_congestion'].dropna(),
    results_df['gabm_congestion'].dropna(),
], labels=['SFM-Core', 'Evac-Baseline', 'Surrogate', 'Full-GABM'])
axes[1].set_title('Congestion Index near door (lower is better)')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()



**Suggested talking points:**

1. We separated effects into mechanics, behavior mapping, and full GABM.
2. We used paired seeds for fairness and reported variability (not just one run).
3. If surrogate ? full GABM, gains are likely from structured behavior mapping, not necessarily LLM-specific generation.
4. If full GABM clearly outperforms both surrogate and non-behavior variants, it supports the extra model complexity.



**Pros and Cons of This New Experimentation**

### Pros
- More causal than simple baseline vs behavioral comparison.
- Uses same NTU data and code stack (low setup overhead).
- Produces clearer engineering trade-off evidence for model complexity decisions.

### Cons
- Still depends on generated behavior grid (external validity limits remain).
- Small seed count (5) is quick but lower confidence than 20+.
- Surrogate model may reflect biases already present in generated labels.

### Recommendation
Use this as a robust interim benchmark, then scale seeds and add real calibration data before operational claims.

